# =============================== # FEATURE ENGINEERING NOTEBOOK # ===============================

## Objectives

- Understand the distribution of numerical and categorical variables in the cleaned dataset.
- Identify missing values, skewness, and potential outliers that may affect modelling.
- Explore the distribution of the target variable (SalePrice) and determine whether transformations (e.g. log) are needed.
- Investigate relationships between predictors and SalePrice using correlation and visual analysis.
Generate insights that guide feature engineering and model selection.

## Inputs
* Cleaned dataset: 
    - `outputs/datasets/cleaned/CleanedData.csv`

## Outputs
- Transformed TrainSet and TestSet
- List of applied transformers:
    - Ordinal categorical encoding
    - Log/Power/Box-Cox transformations
    - Winsorization (IQR)
    - Smart correlation selection


### Import Cell

In [1]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats


from feature_engine.encoding import OneHotEncoder
from feature_engine.encoding import OrdinalEncoder
from feature_engine.selection import SmartCorrelatedSelection
from feature_engine.outliers import Winsorizer
from pathlib import Path

sns.set(style="whitegrid")

---

## Change working directory

We need to set the current working directory to the parent folder for consistency.

In [2]:
BASE_DIR = Path(r"C:\Users\david\Portfolio 5\heritage-housing")
os.chdir(BASE_DIR)
print("Working directory set to:", os.getcwd())

Working directory set to: C:\Users\david\Portfolio 5\heritage-housing


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'C:\\Users\\david\\Portfolio 5\\heritage-housing'

## Load Cleaned Data

We load the cleaned dataset and split it into Train and Test sets.



In [ ]:
from sklearn.model_selection import train_test_split

# Load data
data_path = BASE_DIR / "outputs/datasets/cleaned/CleanedData.csv"
df = pd.read_csv(data_path)


In [12]:
print("Missing in loaded file:", df.isna().sum().sum())
print(df.isna().sum().sort_values(ascending=False).head(20))
print(os.getcwd())
print(os.path.abspath("outputs/datasets/cleaned/CleanedData.csv"))

Missing in loaded file: 418
GarageFinish    235
BsmtFinType1    145
BsmtExposure     38
2ndFlrSF          0
BedroomAbvGr      0
1stFlrSF          0
BsmtFinSF1        0
BsmtUnfSF         0
GarageArea        0
GarageYrBlt       0
GrLivArea         0
KitchenQual       0
LotArea           0
LotFrontage       0
MasVnrArea        0
OpenPorchSF       0
OverallCond       0
OverallQual       0
TotalBsmtSF       0
YearBuilt         0
dtype: int64
C:\Users\david\Portfolio 5\heritage-housing
C:\Users\david\Portfolio 5\heritage-housing\outputs\datasets\cleaned\CleanedData.csv


In [13]:

print("Missing before split:", df.isna().sum().sum())

# Split into Train and Test sets
TrainSet, TestSet = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

TrainSet = TrainSet.copy()
TestSet = TestSet.copy()

# Missing values check
print("Missing in full dataset:", df.isna().sum().sum())
print("Missing in TrainSet:", TrainSet.isna().sum().sum())
print("Missing in TestSet:", TestSet.isna().sum().sum())

print("\nTop missing columns (TrainSet):")
print(TrainSet.isna().sum().sort_values(ascending=False).head(20))

# Separate predictors and target
X_train = TrainSet.drop("SalePrice", axis=1)
y_train = TrainSet["SalePrice"]

X_test = TestSet.drop("SalePrice", axis=1)
y_test = TestSet["SalePrice"]

print("\nShapes:")
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

Missing before split: 418
Missing in full dataset: 418
Missing in TrainSet: 330
Missing in TestSet: 88

Top missing columns (TrainSet):
GarageFinish    186
BsmtFinType1    116
BsmtExposure     28
2ndFlrSF          0
BedroomAbvGr      0
1stFlrSF          0
BsmtFinSF1        0
BsmtUnfSF         0
GarageArea        0
GarageYrBlt       0
GrLivArea         0
KitchenQual       0
LotArea           0
LotFrontage       0
MasVnrArea        0
OpenPorchSF       0
OverallCond       0
OverallQual       0
TotalBsmtSF       0
YearBuilt         0
dtype: int64

Shapes:
(1168, 21) (1168,)
(292, 21) (292,)


---

## Helper Functions

These functions help check missing values and generate diagnostic plots for numeric and categorical features.

In [14]:
# Check missing values
def check_missing_values(df):
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("* No missing values found.")
    else:
        print("* Missing values found:")
        print(missing)

# Numeric diagnostic plots
def diagnostic_plots(df, col):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    sns.histplot(df[col], kde=True, ax=axes[0])
    stats.probplot(df[col], dist="norm", plot=axes[1])
    sns.boxplot(x=df[col], ax=axes[2])
    axes[0].set_title("Histogram")
    axes[1].set_title("QQ Plot")
    axes[2].set_title("Boxplot")
    fig.suptitle(col, fontsize=18, y=1.05)
    plt.tight_layout()
    plt.show()

# Categorical diagnostic plots
def diagnostic_plots_cat(df, col):
    plt.figure(figsize=(4,3))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(col)
    plt.xticks(rotation=90)
    plt.show()

# ===============================
## Phase 1: Exploration (Analysis only)
# ===============================

## Identity Variables for Feature Engineering

### Phase 1: Transformation Exploration & Selection

In [15]:
from sklearn.preprocessing import PowerTransformer

numeric_skewed = ['GrLivArea', 'LotArea', 'TotalBsmtSF', '1stFlrSF', 'BsmtFinSF1']

df_engineering = X_train[numeric_skewed].copy()

check_missing_values(df_engineering)

* No missing values found.


### Numerical Transformation Evaluation (Skewness Comparison)

In [16]:
analysis = []

for col in numeric_skewed:
    row = {"Feature": col}

    row["Original Skew"] = X_train[col].skew()
    row["Log1p Skew"] = np.log1p(X_train[col]).skew()

    analysis.append(row)

skew_df = pd.DataFrame(analysis)
skew_df

,Feature,Original Skew,Log1p Skew
0,GrLivArea,1.425139,0.007943
1,LotArea,11.958088,-0.012599
2,TotalBsmtSF,1.723881,-5.274632
3,1stFlrSF,1.422162,0.026976
4,BsmtFinSF1,1.862132,-0.622601


### Ordinal Categorical Variables

- `KitchenQual`, `ExterQual`, `ExterCond`, `BsmtQual`, `BsmtCond`, `GarageQual`, `GarageCond`, `FireplaceQu`  
These features have a natural order, so we will encode them numerically.

### Numeric Variables for Transformation:
- `GrLivArea`, `LotArea`, `TotalBsmtSF`, `1stFlrSF`, `BsmtFinSF1`  
These features are right-skewed or have outliers and could potentially benefit from log/power/Box-Cox transformations.

### Variables for Correlation-Based Reduction:
- All numeric features  
To reduce multicollinearity and  thus keep only the most informative features.

---

In [17]:
ordinal_vars = [
    'KitchenQual', 'ExterQual', 'ExterCond',
    'BsmtQual', 'BsmtCond',
    'GarageQual', 'GarageCond',
    'FireplaceQu'
]

boxcox_vars = ['GrLivArea', '1stFlrSF']

yeojohnson_vars = ['LotArea', 'TotalBsmtSF', 'BsmtFinSF1']

# ===============================
## Phase 2: Final Pipeline (Model Ready)
# ===============================

### Categorical Encoding (One-Hot)
Ordinal variables were encoded earlier. Remaining nominal categorical variables are one-hot encoded here.


In [18]:
# Automatically encode all object-type categorical columns
ohe = OneHotEncoder(variables=None, drop_last=False)

X_train = ohe.fit_transform(X_train)
X_test = ohe.transform(X_test)

print("* One-hot encoding applied!")
print(X_train.shape, X_test.shape)

c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\feature_engine\variable_handling\_variable_type_checks.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  var = pd.to_datetime(column, utc=True)
c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\feature_engine\variable_handling\_variable_type_checks.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  var = pd.to_datetime(column, utc=True)
c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\feature_engine\variable_handling\_variable_type_checks.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, plea

ValueError: Some of the variables in the dataset contain NaN. Check and remove those before using this transformer.

### Smart Correlation 

- Spearman correlation is used as the method because the dataset contains skewed numerical variables with outliers, and we are interested in monotonic relationships rather than strictly linear dependencies.
- Correlation analysis is used to identify redundant features and reduce multicollinearity before modeling.

In [ ]:
corr_sel = SmartCorrelatedSelection(
    variables=None,
    method="spearman",
    threshold=0.6,
    selection_method="variance"
)

X_train = corr_sel.fit_transform(X_train)
X_test = corr_sel.transform(X_test)

dropped_features = corr_sel.features_to_drop_
print(dropped_features)

print("* Smart correlation selection applied!")
print("Dropped features:", corr_sel.features_to_drop_)

In [ ]:
print(X_train.shape, X_test.shape)
print(list(X_train.columns) == list(X_test.columns))

In [ ]:
print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

## Apply Feature Engineering Transformers

In [ ]:
from sklearn.preprocessing import PowerTransformer

# Box-Cox selected for strongly right-skewed strictly positive variables
boxcox_vars = [c for c in boxcox_vars if c in X_train.columns]


# Apply Box-Cox
pt_boxcox = PowerTransformer(method="box-cox")
X_train[boxcox_vars] = pt_boxcox.fit_transform(X_train[boxcox_vars])
X_test[boxcox_vars] = pt_boxcox.transform(X_test[boxcox_vars])

# Yeo-Johnson selected for skewed variables (allows zeros)
pt_yeo = PowerTransformer(method="yeo-johnson")
X_train[yeojohnson_vars] = pt_yeo.fit_transform(X_train[yeojohnson_vars])
X_test[yeojohnson_vars] = pt_yeo.transform(X_test[yeojohnson_vars])

print(X_train[boxcox_vars].min())
print("* Power transformations applied!")


### Outlier Handling (Winsorization)
Apply IQR-based Winsorization to selected numeric features.

---

In [ ]:
from feature_engine.outliers import Winsorizer

winsor_vars = ['GrLivArea', 'LotArea', 'TotalBsmtSF']

winsor = Winsorizer(
    capping_method='iqr',
    tail='both',
    fold=1.5,
    variables=winsor_vars
)

X_train = winsor.fit_transform(X_train)
X_test = winsor.transform(X_test)

print("* Winsorization applied!")

In [ ]:
# Evaluate Winsorization visually (Train vs Test consistency check)
for col in winsor_vars:
    diagnostic_plots(X_train, col)
    diagnostic_plots(X_test, col)

### Numerical Transformations 
- Apply Power Transformations to reduce skewness
- Box-Cox applied to strictly positive features
- Yeo-Johnson applied to all remaining skewed features 

### Transformation Summary
Final selected transformations are implemented in Phase 2 pipeline.

In [ ]:
print("=== FINAL FEATURE ENGINEERING SUMMARY ===")

print("\nOrdinal encoded variables:")
print(ordinal_vars)

print("\nBox-Cox variables:")
print(boxcox_vars)

print("\nYeo-Johnson variables:")
print(yeojohnson_vars)

print("\nWinsorized variables:")
print(winsor_vars)

print("\nCorrelation-based dropped features:")
print(corr_sel.features_to_drop_)

# ===============================
## Phase 3: Final Outputs
# ===============================

In [ ]:
# Rebuild Train/Test sets for export
TrainSet_final = X_train.copy()
TrainSet_final["SalePrice"] = y_train.values

TestSet_final = X_test.copy()
TestSet_final["SalePrice"] = y_test.values

print(TrainSet_final.shape, TestSet_final.shape)
print(y_train.shape, y_test.shape)

In [ ]:
os.makedirs("outputs/datasets/processed", exist_ok=True)
TrainSet_final.to_csv("outputs/datasets/processed/TrainSet.csv", index=False)
TestSet_final.to_csv("outputs/datasets/processed/TestSet.csv", index=False)